In [2]:
from grain.sources import ArrayRecordDataSource
from pathlib import Path
import msgpack
import numpy as np
from functools import lru_cache

In [3]:
def decode_record(raw: bytes) -> dict:
    record = msgpack.unpackb(raw, raw=False)
    m = np.frombuffer(record["m"], dtype=np.float32).reshape(record["shape"])
    Ms = record["params"]["Ms"]
    H_ext = np.asarray(record["H_ext"], dtype=np.float32)
    return {"m": m, "Ms": Ms, "H_ext": H_ext}

In [4]:
class FramePairDataSource:
    def __init__(self, source, T, decode_fn, delta=1):
        self._src = source
        self._decode = decode_fn  # bytes -> np.ndarray (t, c, h, w)
        self._delta = delta

        self._index = []
        for rec in range(len(self._src)):
            for f in range(T - delta):
                self._index.append((rec, f))

    def __len__(self):
        return len(self._index)

    @lru_cache(32)
    def _decode_cached(self, rec):
        return self._decode(self._src[rec])

    def index_dict(self, traj, f):
        return {
            "m0": traj["m"][f],
            "m1": traj["m"][f + self._delta],
            "Ms": traj["Ms"],
            "H_ext": traj["H_ext"],
        }

    def __getitem__(self, idx: int):
        rec, f = self._index[idx]
        traj = self._decode_cached(rec)  # (T, C, H, W)
        return self.index_dict(traj, f)

In [7]:
class TrajectoryDataSource:
    def __init__(self, source, decode_fn):
        self._src = source
        self._decode = decode_fn

    def __len__(self):
        return len(self._src)

    @lru_cache(32)
    def _decode_cached(self, idx):
        return self._decode(self._src[idx])

    def __getitem__(self, idx: int):
        return self._decode_cached(idx)

In [8]:
data_dir = Path("../../micromagnetic-data/data3/dynamics/small")
split = "train"

shards = [str(p) for p in (data_dir / split).glob("*.arrayrecord")]
source = ArrayRecordDataSource(shards)

pair_source = FramePairDataSource(source, T=101, decode_fn=decode_record)
traj_source = TrajectoryDataSource(source=source, decode_fn=decode_record)

In [11]:
from grain import MapDataset

ds = MapDataset.source(source=pair_source)
ds = ds.batch(32).to_iter_dataset()

In [12]:
for batch in ds:
    for k, v in batch.items():
        print(k, v.shape)

H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext (32, 3)
Ms (32,)
m0 (32, 256, 256, 1, 3)
m1 (32, 256, 256, 1, 3)
H_ext 

KeyboardInterrupt: 